# ACDC (slow-mode) circuit discovery via AutoCircuit

**Notebook 16 — openinterpretability-web**

This notebook runs **ACDC** (Automated Circuit Discovery) and **attribution-patching-as-edge-pruning** on a small transformer via the [`auto-circuit`](https://github.com/UFO-101/auto-circuit) library — the current practitioner-default implementation, which the original ACDC repo [ArthurConmy/Automatic-Circuit-Discovery](https://github.com/ArthurConmy/Automatic-Circuit-Discovery) (frozen 2024-10) now officially points users to.

## Framing: fast path vs. slow path

- **Notebook 14** (fast path) — ships **AtP\*** for *node* attribution. Seconds to minutes on modern hardware. Good for day-to-day circuit sketching.
- **Notebook 16 (this one, slow path)** — runs the original **peer-reviewed ACDC algorithm** (Conmy et al., NeurIPS 2023) for **edge**-level discovery. Orders of magnitude slower, but is the academic-rigor reference. Use it as an independent cross-check on whatever AtP\* in notebook 14 suggests.

## Primary sources

- Conmy, Mavor-Parker, Lynch, Heimersheim, Garriga-Alonso. *Towards Automated Circuit Discovery for Mechanistic Interpretability.* NeurIPS 2023. [arxiv:2304.14997](https://arxiv.org/abs/2304.14997)
- Original repo (frozen): [github.com/ArthurConmy/Automatic-Circuit-Discovery](https://github.com/ArthurConmy/Automatic-Circuit-Discovery) — directs to AutoCircuit
- Practitioner default: [github.com/UFO-101/auto-circuit](https://github.com/UFO-101/auto-circuit) — ACDC + attribution patching + edge pruning in one API
- Syed, Rager, Conmy. *Attribution Patching Outperforms Automated Circuit Discovery.* [arxiv:2310.10348](https://arxiv.org/abs/2310.10348) — finding: AtP-as-pruning generally dominates ACDC at equal edge budgets.

## Honest caveats up front

- **No SAE-feature edges upstream.** `auto-circuit` operates on **residual-stream activation nodes** (attention heads, MLP layers, token positions). It does not, out of the box, prune edges defined over SAE features. If you want SAE-feature circuits you run ACDC at the activation level, then **manually map** high-weight nodes back to the SAE features that fire at those sites.
- **Model size.** `auto-circuit` sits on top of [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens), so the supported model list is small (GPT-2 family, Pythia, Llama-ish forks, Qwen via community patches). We default to `pythia-160m` here because it is small enough for a Colab T4 and is officially supported.
- **Runtime.** Full ACDC on GPT-2-small for the IOI task is ~1–2h on a T4, ~20 min on an A100. The attribution-patching-as-pruning variant from the same library is ~5 min.
- **Expect AtP-pruning to win.** Per Syed et al. 2023, attribution-patching-as-pruning matches or beats ACDC at equal edge budgets in most benchmarks. We report both so you can see the gap yourself.


In [ ]:
# Cell 2 — install
# auto-circuit pulls in transformer_lens; we pin nothing here, rely on the library's own deps.
# If the install fails on Colab (TransformerLens sometimes breaks on new torch), see the fallback
# cell near the end of this notebook for a pure-TransformerLens custom edge-pruning loop.

%pip install -q transformers accelerate safetensors huggingface_hub matplotlib
%pip install -q transformer_lens
%pip install -q auto-circuit

import sys, importlib
for mod in ["transformer_lens", "auto_circuit"]:
    try:
        importlib.import_module(mod)
        print(f"[ok] {mod} imported")
    except Exception as e:
        print(f"[FAIL] {mod}: {e}")
        print("See the fallback cell near the end of this notebook.")


## Config

We default to **`pythia-160m`** on the **IOI** (Indirect Object Identification) task, the canonical benchmark from the ACDC paper. Swap `BASE_MODEL` to `'gpt2'` or `'gpt2-small'` if you want to replicate the Conmy et al. numbers exactly.


In [ ]:
# Cell 4 — config
from dataclasses import dataclass

@dataclass
class Config:
    # Use a TransformerLens-supported model. 'pythia-160m' fits a T4; 'gpt2' replicates the paper.
    BASE_MODEL: str = "pythia-160m"
    # Canonical IOI benchmark from the ACDC paper.
    TASK: str = "ioi"
    # Target size of the final discovered circuit (#edges kept).
    EDGE_COUNT_TARGET: int = 20
    # Metric driving ACDC's edge-keep decisions. 'logit_diff' is the IOI default.
    METRIC: str = "logit_diff"
    # ACDC threshold tau: smaller = more edges kept, larger = more aggressive pruning.
    # 0.01 is a middle-of-the-road value from the paper's sweeps.
    TAU: float = 0.01
    # HF repo to upload the exported circuit.json. Same SAE repo used by notebook 15.
    HF_SAE_REPO: str = "caiovicentino1/openinterp-web-sae"
    DEVICE: str = "cuda"

cfg = Config()
print(cfg)


## Load model via TransformerLens

`auto-circuit` requires a `HookedTransformer`. We load in `float32` because ACDC's finite-difference patching is sensitive to numerical noise — `bfloat16` can flip the keep/drop decision on edges near the threshold.


In [ ]:
# Cell 6 — load model
import torch
from transformer_lens import HookedTransformer

device = cfg.DEVICE if torch.cuda.is_available() else "cpu"
print(f"[info] loading {cfg.BASE_MODEL} on {device} in float32")

model = HookedTransformer.from_pretrained(
    cfg.BASE_MODEL,
    dtype=torch.float32,  # ACDC needs fp32 for stable keep/drop decisions
)
model = model.to(device)
model.eval()

# auto-circuit needs these toggled so every attention-head / MLP edge is individually hookable.
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)
model.set_use_hook_mlp_in(True)

print(f"[ok] n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, d_model={model.cfg.d_model}")


## Load the benchmark task dataset

The IOI task pairs a *clean* prompt (`"When John and Mary went to the store, John gave a drink to"` → `Mary`) with a *corrupt* prompt where the second name is swapped. ACDC measures how much each edge contributes to the clean-minus-corrupt logit difference.

`auto-circuit` ships the IOI task pre-built under `auto_circuit.tasks.IOI_TOKEN_CIRCUIT_TASK`. If that import path has drifted in your installed version, the fallback below builds a minimal 8-example IOI dataset inline.


In [ ]:
# Cell 8 — load IOI task
task = None
try:
    from auto_circuit.tasks import IOI_TOKEN_CIRCUIT_TASK as task
    print("[ok] loaded auto_circuit.tasks.IOI_TOKEN_CIRCUIT_TASK")
except Exception as e:
    print(f"[warn] could not import packaged IOI task: {e}")
    print("[info] building minimal inline IOI dataset")

if task is None:
    # Minimal inline IOI: 8 clean/corrupt pairs. Real benchmark uses ~100.
    import torch
    from auto_circuit.data import PromptDataset, PromptPairBatch, PromptDataLoader

    clean_prompts = [
        "When John and Mary went to the store, John gave a drink to",
        "When Alice and Bob went to the park, Alice gave a ball to",
        "When Sara and Tim went to the office, Sara gave a pen to",
        "When Liam and Noah went to the pool, Liam gave a towel to",
    ]
    corrupt_prompts = [
        "When John and Mary went to the store, Mary gave a drink to",
        "When Alice and Bob went to the park, Bob gave a ball to",
        "When Sara and Tim went to the office, Tim gave a pen to",
        "When Liam and Noah went to the pool, Noah gave a towel to",
    ]
    # answers: IO token (correct completion) vs S token (subject, the wrong completion)
    io_tokens = [" Mary", " Bob", " Tim", " Noah"]
    s_tokens  = [" John", " Alice", " Sara", " Liam"]

    def tok1(s):
        ids = model.to_tokens(s, prepend_bos=False)[0]
        return ids[0].item()

    clean_ids   = [model.to_tokens(p)[0] for p in clean_prompts]
    corrupt_ids = [model.to_tokens(p)[0] for p in corrupt_prompts]
    answer_ids  = [torch.tensor([tok1(io), tok1(s)]) for io, s in zip(io_tokens, s_tokens)]

    print(f"[ok] inline IOI: {len(clean_prompts)} pairs")
    inline_ioi = dict(
        clean=clean_ids, corrupt=corrupt_ids, answers=answer_ids,
        prompts_clean=clean_prompts, prompts_corrupt=corrupt_prompts,
    )
else:
    inline_ioi = None
    print(f"[ok] task: {task}")


## Run ACDC

`acdc_prune_scores` performs the Conmy et al. algorithm: walk edges in reverse-topological order, for each edge test whether patching it with the corrupt activation changes the metric by more than `TAU`; if not, drop it.

This is the slow step. On `pythia-160m` + 8 prompts + `TAU=0.01`, expect ~2–10 min on a T4 GPU. On `gpt2` + full 100-prompt IOI + same `TAU`, expect 1–2 hours on a T4.


In [ ]:
# Cell 10 — run ACDC
import time

acdc_scores = None
t0 = time.time()
try:
    from auto_circuit.prune_algos.ACDC import acdc_prune_scores
    from auto_circuit.utils.graph_utils import patchable_model

    # wrap HookedTransformer into the patchable graph auto-circuit expects
    pmodel = patchable_model(
        model,
        factorized=True,
        slice_output="last_seq",
        separate_qkv=True,
        device=device,
    )

    if task is not None:
        dataloader = task.train_loader
    else:
        # build a minimal dataloader from the inline dataset
        from auto_circuit.data import PromptPairBatch
        batch = PromptPairBatch(
            key=0,
            batch_diverge_idx=0,
            clean=torch.stack(inline_ioi["clean"]).to(device),
            corrupt=torch.stack(inline_ioi["corrupt"]).to(device),
            answers=[a.to(device) for a in inline_ioi["answers"]],
            wrong_answers=[a.flip(0).to(device) for a in inline_ioi["answers"]],
        )
        dataloader = [batch]

    print(f"[info] running ACDC with tao={cfg.TAU} ... this is the slow step.")
    acdc_scores = acdc_prune_scores(
        model=pmodel,
        dataloader=dataloader,
        official_edges=None,
        tao_exps=[-2],           # tao = 10**-2 = 0.01
        tao_bases=[1],
    )
    print(f"[ok] ACDC done in {time.time()-t0:.1f}s — {sum(1 for v in acdc_scores.values() if abs(v)>0)} non-zero edges")
except Exception as e:
    print(f"[FAIL] ACDC pass errored: {e}")
    print("       Falling back to AtP-pruning only in the next cell.")


## Compare with attribution-patching-as-pruning

Now we run the fast alternative from the same library: **mask-gradient pruning** = attribution patching used as an edge-ranking signal. This is what Syed, Rager, Conmy (2023) showed matches or beats ACDC at equal edge budgets. Runtime is typically 10–100× faster than ACDC.

We plot both faithfulness-vs-edges-kept curves on the same axes so the gap (if any) is visible.


In [ ]:
# Cell 12 — attribution patching as pruning + faithfulness curves
import time, math
import matplotlib.pyplot as plt

atp_scores = None
t0 = time.time()
try:
    from auto_circuit.prune_algos.mask_gradient import mask_gradient_prune_scores
    atp_scores = mask_gradient_prune_scores(
        model=pmodel,
        dataloader=dataloader,
        official_edges=None,
        grad_function="logit",
        answer_function="avg_diff",
        mask_val=0.0,
    )
    print(f"[ok] AtP-pruning done in {time.time()-t0:.1f}s")
except Exception as e:
    print(f"[FAIL] AtP-pruning errored: {e}")

# --- faithfulness-vs-edges-kept ------------------------------------------------
def faithfulness_curve(scores, pmodel, dataloader, edge_counts):
    """Return list of (n_edges, faithfulness) by keeping top-n edges by |score|."""
    from auto_circuit.prune import run_circuits
    from auto_circuit.types import PatchType, AblationType
    out = []
    for n in edge_counts:
        try:
            results = run_circuits(
                model=pmodel,
                dataloader=dataloader,
                test_edge_counts=[n],
                prune_scores=scores,
                patch_type=PatchType.TREE_PATCH,
                ablation_type=AblationType.RESAMPLE,
            )
            # faithfulness ≈ metric(circuit) / metric(full model); run_circuits returns per-n dicts
            val = list(results[n].values())[0]
            out.append((n, float(val)))
        except Exception as e:
            print(f"  [warn] faithfulness at n={n} failed: {e}")
    return out

edge_counts = [2, 5, 10, 20, 50, 100, 200]

fig, ax = plt.subplots(figsize=(7, 4.5))
legend_rows = []

if acdc_scores is not None:
    curve_acdc = faithfulness_curve(acdc_scores, pmodel, dataloader, edge_counts)
    if curve_acdc:
        xs, ys = zip(*curve_acdc)
        ax.plot(xs, ys, marker="o", label="ACDC")
        legend_rows.append("ACDC")

if atp_scores is not None:
    curve_atp = faithfulness_curve(atp_scores, pmodel, dataloader, edge_counts)
    if curve_atp:
        xs, ys = zip(*curve_atp)
        ax.plot(xs, ys, marker="s", label="Attribution Patching (pruning)")
        legend_rows.append("AtP-pruning")

ax.set_xscale("log")
ax.set_xlabel("# edges kept")
ax.set_ylabel(f"faithfulness ({cfg.METRIC})")
ax.set_title(f"ACDC vs AtP-pruning on {cfg.BASE_MODEL} / {cfg.TASK}")
ax.grid(True, alpha=0.3)
if legend_rows:
    ax.legend()
plt.tight_layout()
plt.savefig("/tmp/acdc_vs_atp_faithfulness.png", dpi=120)
plt.show()

print("[info] Per Syed, Rager, Conmy 2023: AtP-pruning typically dominates ACDC at equal edge budgets.")
print("       If your curves show ACDC ahead at very low edge counts, that's also consistent with the paper:")
print("       ACDC can squeeze a few more points of faithfulness into the smallest circuits.")


## Export circuit

We convert `auto-circuit`'s edge list to the same `circuit.json` schema notebook 15 uses, so the Circuit Canvas viewer can render it alongside AtP\* circuits from notebook 14. Each edge becomes a `{source, target, weight}` record; nodes carry `{id, layer, kind}` where `kind` is one of `attn_head`, `mlp`, `resid`.


In [ ]:
# Cell 14 — export circuit.json + upload
import json, os, pathlib

# Pick whichever scores we got, prefer ACDC because that's the point of this notebook.
chosen = ("acdc", acdc_scores) if acdc_scores is not None else ("atp_pruning", atp_scores)
method, scores = chosen
if scores is None:
    raise RuntimeError("Neither ACDC nor AtP-pruning produced scores; cannot export.")

# Flatten scores dict → list of (edge_obj, score) sorted by |score|.
flat = []
for module_name, tensor in scores.items():
    try:
        t = tensor.detach().cpu().flatten()
        for i, v in enumerate(t.tolist()):
            flat.append((module_name, i, float(v)))
    except AttributeError:
        # scalar
        flat.append((module_name, 0, float(tensor)))
flat.sort(key=lambda r: -abs(r[2]))
top = flat[: cfg.EDGE_COUNT_TARGET]

# Build schema-compatible circuit.json (matches notebook 15's output shape)
def parse_node(name):
    # e.g. "blocks.3.attn.hook_result" → (layer=3, kind='attn_head')
    parts = name.split(".")
    layer = None
    for i, p in enumerate(parts):
        if p == "blocks" and i+1 < len(parts) and parts[i+1].isdigit():
            layer = int(parts[i+1])
            break
    if "attn" in name:
        kind = "attn_head"
    elif "mlp" in name:
        kind = "mlp"
    else:
        kind = "resid"
    return layer, kind

nodes_seen = {}
edges = []
for module_name, idx, score in top:
    layer, kind = parse_node(module_name)
    node_id = f"{module_name}#{idx}"
    if node_id not in nodes_seen:
        nodes_seen[node_id] = {"id": node_id, "layer": layer, "kind": kind}
    # edges in auto-circuit's factorized graph are indexed as (src_module, dst_module);
    # for the viewer we just emit a self-loop-style edge with the score as weight.
    edges.append({
        "source": node_id,
        "target": node_id,
        "weight": score,
    })

circuit = {
    "schema_version": "1.0",
    "source_notebook": "16_autocircuit_acdc",
    "method": method,
    "base_model": cfg.BASE_MODEL,
    "task": cfg.TASK,
    "metric": cfg.METRIC,
    "tau": cfg.TAU if method == "acdc" else None,
    "n_edges_kept": len(edges),
    "nodes": list(nodes_seen.values()),
    "edges": edges,
}

out_path = pathlib.Path("/tmp/circuit.json")
out_path.write_text(json.dumps(circuit, indent=2))
print(f"[ok] wrote {out_path} — {len(edges)} edges, {len(nodes_seen)} nodes")
print(json.dumps(circuit, indent=2)[:800], "...")

# Upload to HF (same repo as notebook 15's circuit output)
try:
    from huggingface_hub import HfApi, login
    import os as _os
    tok = _os.environ.get("HF_TOKEN")
    if tok:
        login(token=tok)
        api = HfApi()
        api.upload_file(
            path_or_fileobj=str(out_path),
            path_in_repo=f"circuits/nb16_{method}_{cfg.BASE_MODEL.replace('/', '_')}_circuit.json",
            repo_id=cfg.HF_SAE_REPO,
            repo_type="model",
        )
        print(f"[ok] uploaded to {cfg.HF_SAE_REPO}")
    else:
        print("[skip] set HF_TOKEN env var to auto-upload; local file is ready at /tmp/circuit.json")
except Exception as e:
    print(f"[warn] HF upload skipped: {e}")


## Fallback: pure TransformerLens custom edge-pruning loop

If the `auto-circuit` install failed (most common reason on Colab: a TransformerLens version mismatch with the current torch), the cell below runs a minimal ACDC-style loop using only TransformerLens' `run_with_hooks`. It is **not** a drop-in replacement — it works at the **attention-head** granularity rather than full edge-level — but it lets you finish the notebook and produce a `circuit.json` that renders in the viewer.

**Algorithm (head-level ACDC)**:

1. Run model on clean and corrupt prompts, record baseline logit_diff.
2. For each (layer, head) pair, patch the head's output with the corrupt activation.
3. If `|logit_diff_patched − logit_diff_clean| < TAU`, the head is redundant → drop it.
4. Kept heads = your circuit.

This is what the original ACDC paper actually does; `auto-circuit` generalizes it to arbitrary edges.


In [ ]:
# Cell 16 — fallback: pure TransformerLens head-level ACDC
# Only run this cell if the auto-circuit cells above failed.

RUN_FALLBACK = False  # set True to execute

if RUN_FALLBACK:
    import torch, json, pathlib
    from functools import partial

    clean_prompts = [
        "When John and Mary went to the store, John gave a drink to",
        "When Alice and Bob went to the park, Alice gave a ball to",
    ]
    corrupt_prompts = [
        "When John and Mary went to the store, Mary gave a drink to",
        "When Alice and Bob went to the park, Bob gave a ball to",
    ]
    io_answers = [" Mary", " Bob"]
    s_answers  = [" John", " Alice"]

    def tok(s):
        return model.to_tokens(s, prepend_bos=False)[0, 0].item()

    io_ids = torch.tensor([tok(a) for a in io_answers], device=device)
    s_ids  = torch.tensor([tok(a) for a in s_answers],  device=device)

    def logit_diff(logits):
        # logits: [batch, seq, vocab]; take last position
        last = logits[:, -1, :]
        return (last[torch.arange(len(io_ids)), io_ids] - last[torch.arange(len(s_ids)), s_ids]).mean()

    clean_tokens   = model.to_tokens(clean_prompts).to(device)
    corrupt_tokens = model.to_tokens(corrupt_prompts).to(device)

    # cache corrupt activations for every head output
    _, corrupt_cache = model.run_with_cache(corrupt_tokens)
    baseline_clean = logit_diff(model(clean_tokens)).item()
    print(f"[info] baseline logit_diff (clean): {baseline_clean:.4f}")

    def patch_head(z, hook, layer, head):
        # z shape: [batch, seq, n_heads, d_head]
        z[:, :, head, :] = corrupt_cache[f"blocks.{layer}.attn.hook_z"][:, :, head, :]
        return z

    kept = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            patched_logits = model.run_with_hooks(
                clean_tokens,
                fwd_hooks=[(f"blocks.{layer}.attn.hook_z", partial(patch_head, layer=layer, head=head))],
            )
            delta = abs(logit_diff(patched_logits).item() - baseline_clean)
            if delta >= cfg.TAU:
                kept.append({"layer": layer, "head": head, "delta": delta})

    kept.sort(key=lambda r: -r["delta"])
    kept = kept[: cfg.EDGE_COUNT_TARGET]
    print(f"[ok] fallback kept {len(kept)} heads")

    circuit = {
        "schema_version": "1.0",
        "source_notebook": "16_autocircuit_acdc",
        "method": "acdc_fallback_head_level",
        "base_model": cfg.BASE_MODEL,
        "task": cfg.TASK,
        "metric": cfg.METRIC,
        "tau": cfg.TAU,
        "n_edges_kept": len(kept),
        "nodes": [{"id": f"L{r['layer']}H{r['head']}", "layer": r["layer"], "kind": "attn_head"} for r in kept],
        "edges": [{"source": f"L{r['layer']}H{r['head']}", "target": f"L{r['layer']}H{r['head']}", "weight": r["delta"]} for r in kept],
    }
    pathlib.Path("/tmp/circuit.json").write_text(json.dumps(circuit, indent=2))
    print("[ok] fallback circuit.json written to /tmp/circuit.json")
else:
    print("[skip] fallback not requested; set RUN_FALLBACK=True above to enable.")


## What to do with the exported circuit

- Open it in the **Circuit Canvas** viewer shipped with notebook 15 — the `schema_version` and field names match.
- Compare it visually against the AtP\*-derived circuit from notebook 14. Overlap = cross-method agreement (high confidence). Disagreement = worth investigating; one of the two methods is missing something.
- To lift this to **SAE-feature edges** rather than attention-head edges: for each high-weight node, run the SAE from notebooks 4/12 at that layer, record which features fire on the task examples, and annotate the circuit nodes with their top-k SAE features. This is the manual-mapping workflow referenced in the framing note at the top — auto-circuit does not do it for you.

### References

- Conmy et al. 2023, *Towards Automated Circuit Discovery for Mechanistic Interpretability*, NeurIPS. arxiv:2304.14997
- Syed, Rager, Conmy 2023, *Attribution Patching Outperforms Automated Circuit Discovery*, arxiv:2310.10348
- [github.com/UFO-101/auto-circuit](https://github.com/UFO-101/auto-circuit) — library used in this notebook
- [github.com/ArthurConmy/Automatic-Circuit-Discovery](https://github.com/ArthurConmy/Automatic-Circuit-Discovery) — original ACDC reference implementation (frozen)
